In [1]:

# IMPORTS


import os
import numpy as np

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

from sklearn.metrics.pairwise import cosine_similarity

from langchain.text_splitter import (
    RecursiveCharacterTextSplitter
)

from docling.document_converter import DocumentConverter

from docx import Document



/Users/anushka/rag_env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/anushka/rag_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

# UPLOAD FILE 


file_path = input("Enter file path: ").strip()

In [5]:

# DOCUMENT PARSER


def parse_document(file_path):

    ext = os.path.splitext(file_path)[1].lower()

    # PDF

    if ext == ".pdf":

        print("\nParsing PDF with Docling...\n")

        converter = DocumentConverter()

        result = converter.convert(file_path)

        markdown_text = result.document.export_to_markdown()

        return markdown_text

  
    # TXT

    elif ext == ".txt":

        print("\nReading TXT file...\n")

        with open(file_path, "r", encoding="utf-8") as f:

            return f.read()


    # DOCX

    elif ext == ".docx":

        print("\nReading DOCX file...\n")

        doc = Document(file_path)

        text = []

        # paragraphs
        for para in doc.paragraphs:

            if para.text.strip():

                text.append(para.text)

        # tables
        for table in doc.tables:

            text.append("\nTABLE:\n")

            for row in table.rows:

                row_text = []

                for cell in row.cells:

                    row_text.append(cell.text.strip())

                text.append(" | ".join(row_text))

            text.append("\n")

        return "\n".join(text)



    # UNSUPPORTED

    else:

        raise ValueError(
            "Unsupported file format. Use PDF, TXT, or DOCX."
        )
    

In [6]:

# PARSE DOCUMENT


document_text = parse_document(file_path)

print("\n================ DOCUMENT PREVIEW ================\n")

print(document_text[:3000])


Parsing PDF with Docling...


================ DOCUMENT PREVIEW ================

OVERED BY LIFT

MANIPAL

UNIVERSITY JAIPUR

A PBL-IV (AIM2270) Synopsis On

## CRASH ALERT DETECTION

Submitted to Manipal University Jaipur

Towards the partial fulfillment for the Award of the Degree of

## B. Tech Computer Science and Engineering (Artificial Intelligence and Machine Learning)

In Computers Science and Engineering (AIML)

2023-2027

By

Name of the Candidates: Abhay Negi &amp; Anushka

Registration Numbers: 23FE10CAI00560 &amp; 23FE10CAI00419.

<!-- image -->

Under the guidance of

## Dr. Shivendra Dubey

## Department of Artificial Intelligence and Machine Learning

## Manipal University Jaipur Jaipur, Rajasthan

## I. Introduction to Problem

Road accidents are a major public safety issue and often lead to serious injuries or fatalities. One of the key reasons accidents become more severe is the delay in providing timely medical assistance . In many situations, victims are unable to

In [7]:

# SAVE AS MARKDOWN FILE


md_file_path = "/Users/anushka/Desktop/parsed_output.md"

with open(md_file_path, "w", encoding="utf-8") as f:

    f.write(document_text)

print(f"\nMarkdown file saved as: {md_file_path}")


Markdown file saved as: /Users/anushka/Desktop/parsed_output.md


In [8]:
# CHUNKING

from langchain.text_splitter import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter
)

print("\nCreating markdown-aware chunks...\n")

# Split by markdown headers
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

md_header_splits = markdown_splitter.split_text(document_text)

# Extract content
md_chunks = [doc.page_content for doc in md_header_splits]

# Secondary recursive splitting
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200
)

chunks = []

for chunk in md_chunks:

    split_chunks = recursive_splitter.split_text(chunk)

    chunks.extend(split_chunks)

print(f"\nTotal Chunks Created: {len(chunks)}")

print("\n================ SAMPLE CHUNK ================\n")

print(chunks[0])


Creating markdown-aware chunks...


Total Chunks Created: 26

================ SAMPLE CHUNK ================

OVERED BY LIFT  
MANIPAL  
UNIVERSITY JAIPUR  
A PBL-IV (AIM2270) Synopsis On


In [9]:
from rank_bm25 import BM25Okapi

# BM25 KEYWORD INDEX

tokenized_chunks = [
    chunk.lower().split()
    for chunk in chunks
]

bm25 = BM25Okapi(tokenized_chunks)

print("BM25 index created successfully!")

BM25 index created successfully!


In [10]:

# PREVIEW CHUNKS


for i, chunk in enumerate(chunks[:3]):

    print(f"\n================ CHUNK {i+1} ================\n")

    print(chunk)


================ CHUNK 1 ================

OVERED BY LIFT  
MANIPAL  
UNIVERSITY JAIPUR  
A PBL-IV (AIM2270) Synopsis On

================ CHUNK 2 ================

Submitted to Manipal University Jaipur  
Towards the partial fulfillment for the Award of the Degree of

================ CHUNK 3 ================

In Computers Science and Engineering (AIML)  
2023-2027  
By  
Name of the Candidates: Abhay Negi &amp; Anushka  
Registration Numbers: 23FE10CAI00560 &amp; 23FE10CAI00419.  
<!-- image -->  
Under the guidance of


In [11]:

# LOAD EMBEDDING MODEL


print("\nLoading embedding model...\n")

embedding_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

print("Embedding model loaded.")


Loading embedding model...

Embedding model loaded.


In [12]:

# GENERATE CHUNK EMBEDDINGS


print("\nGenerating embeddings...\n")

chunk_embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True
)

print("Embeddings created.")

print("\nEmbedding Shape:\n")

print(chunk_embeddings.shape)


Generating embeddings...

Embeddings created.

Embedding Shape:

(26, 768)


In [13]:

# CREATE FAISS VECTOR INDEX
import faiss
embedding_dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(
    embedding_dimension
)

index.add(
    np.array(chunk_embeddings).astype("float32")
)

print("FAISS index created successfully!")

print("Total vectors in index:", index.ntotal)

FAISS index created successfully!
Total vectors in index: 26


In [14]:

# USER QUERY


query = input("\nAsk your question: ")

In [15]:

# QUERY EMBEDDING


query_embedding = embedding_model.encode(
    query,
    convert_to_numpy=True
)

In [16]:

# ============================================================
# HYBRID RETRIEVAL
# FAISS + BM25
# ============================================================

from IPython.display import display, Markdown

print("\nPerforming Hybrid Retrieval...\n")


# ============================================================
# QUERY VECTOR
# ============================================================

query_vector = np.array(
    [query_embedding]
).astype("float32")


# ============================================================
# FAISS SEARCH
# ============================================================

top_k = 20

distances, faiss_indices = index.search(
    query_vector,
    top_k
)


# ============================================================
# CONVERT DISTANCE → SIMILARITY
# ============================================================

semantic_scores = 1 / (1 + distances[0])


# ============================================================
# BM25 SEARCH
# ============================================================

tokenized_query = query.lower().split()

bm25_scores = bm25.get_scores(
    tokenized_query
)

bm25_scores = (
    bm25_scores
    / bm25_scores.max()
)


# ============================================================
# COMBINE SCORES
# ============================================================

hybrid_results = []

for i, idx in enumerate(faiss_indices[0]):

    semantic_score = semantic_scores[i]

    keyword_score = bm25_scores[idx]

    hybrid_score = (
        0.7 * semantic_score
        +
        0.3 * keyword_score
    )

    hybrid_results.append(
        (
            hybrid_score,
            chunks[idx]
        )
    )


# ============================================================
# SORT RESULTS
# ============================================================

hybrid_results = sorted(
    hybrid_results,
    key=lambda x: x[0],
    reverse=True
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

retrieved_chunks = []

print("\n================ RETRIEVED CHUNKS ================\n")

for score, chunk in hybrid_results:

    retrieved_chunks.append(chunk)

    print(f"\nHybrid Score: {score:.4f}\n")

    display(Markdown(chunk))

    print("\n--------------------------------------------------")


Performing Hybrid Retrieval...


================ RETRIEVED CHUNKS ================


Hybrid Score: 0.6396



The comparative analysis shows that most existing systems focus mainly on accident detection or accident prediction individually. However, they lack an integrated system that combines interactive mapping, accident hotspot visualization, machine learning crash risk prediction, and route safety recommendation.  
The proposed system addresses this gap by integrating machine learning analysis with an interactive map-based interface, enabling users to visualize accident-prone areas, predict crash risk, and choose safer travel routes.  
CERTIFICATE  
This is to certify that the project entitled 'Al Model for Financial Market Prediction'  
is a Bonafide work carried out as part of the course AIM2270, under my guidance from Jan 2025 to May 2025 by Anushka ( 23FE10CAI00419) , student of B. Tech  
(hons.) Computer Science and Engineering (AIML), 4th Semester at the Department of Artificial Intelligence and Machine Learning, Manipal University Jaipur, during the academic semester 4* in partial fulfilment of the requirements for the award of the degree of Bachelor of Technology in Computer Science and Engineering (AIML), at MUJ. Jaipur.  
Dept Supervisor


--------------------------------------------------

Hybrid Score: 0.6099



Dept Supervisor  
Project Guide, Dept of (Name of the Dept.)  
Manipal University Jaipur  
<!-- image -->  
HoD  
HOD Name  
Department of AIML  
Manipal University Jaipur  
AIML  
HOD, Dept of (Name of the De/aginal University Jaipur  
<!-- image -->  
Date: 18/04/25  
AICS assignment information  
//  
<!-- image -->


--------------------------------------------------

Hybrid Score: 0.5876



...


--------------------------------------------------

Hybrid Score: 0.3302



- High prediction accuracy using machine learning
- Identifies key environmental factors affecting accidents
- Useful for accident risk prediction systems


--------------------------------------------------

Hybrid Score: 0.3297



- High accuracy using deep learning models
- Real-time accident detection from video streams
- Suitable for smart city surveillance systems


--------------------------------------------------


In [17]:

# LOAD RERANKER


print("\nLoading reranker model...\n")

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Reranker loaded.")


Loading reranker model...

Reranker loaded.


In [18]:

# RERANK RESULTS


print("\nReranking retrieved chunks...\n")

pairs = [
    [query, chunk]
    for chunk in retrieved_chunks
]

rerank_scores = reranker.predict(pairs)

reranked_results = sorted(
    zip(rerank_scores, retrieved_chunks),
    reverse=True
)

print("\n================ FINAL RERANKED RESULTS ================\n")

for score, chunk in reranked_results:

    print(f"\nRerank Score: {score:.4f}\n")

    print(chunk[:1500])

    print("\n========================================================")


Reranking retrieved chunks...


================ FINAL RERANKED RESULTS ================


Rerank Score: -6.2425

The comparative analysis shows that most existing systems focus mainly on accident detection or accident prediction individually. However, they lack an integrated system that combines interactive mapping, accident hotspot visualization, machine learning crash risk prediction, and route safety recommendation.  
The proposed system addresses this gap by integrating machine learning analysis with an interactive map-based interface, enabling users to visualize accident-prone areas, predict crash risk, and choose safer travel routes.  
CERTIFICATE  
This is to certify that the project entitled 'Al Model for Financial Market Prediction'  
is a Bonafide work carried out as part of the course AIM2270, under my guidance from Jan 2025 to May 2025 by Anushka ( 23FE10CAI00419) , student of B. Tech  
(hons.) Computer Science and Engineering (AIML), 4th Semester at the Department of Art

In [19]:
# PREPARE CONTEXT

top_chunks = [
    chunk for score, chunk in reranked_results[:8]
]

context = "\n\n".join(top_chunks)

print("\n================ FINAL RETRIEVED CONTEXT ================\n")

for i, chunk in enumerate(top_chunks):

    print(f"\n--- Context Chunk {i+1} ---\n")

    print(chunk[:1500])


================ FINAL RETRIEVED CONTEXT ================


--- Context Chunk 1 ---

The comparative analysis shows that most existing systems focus mainly on accident detection or accident prediction individually. However, they lack an integrated system that combines interactive mapping, accident hotspot visualization, machine learning crash risk prediction, and route safety recommendation.  
The proposed system addresses this gap by integrating machine learning analysis with an interactive map-based interface, enabling users to visualize accident-prone areas, predict crash risk, and choose safer travel routes.  
CERTIFICATE  
This is to certify that the project entitled 'Al Model for Financial Market Prediction'  
is a Bonafide work carried out as part of the course AIM2270, under my guidance from Jan 2025 to May 2025 by Anushka ( 23FE10CAI00419) , student of B. Tech  
(hons.) Computer Science and Engineering (AIML), 4th Semester at the Department of Artificial Intelligence and Mach

In [20]:

# CREATE PROMPT

prompt = f"""
You are a research paper assistant.

Answer ONLY using the provided context.

Format the answer using:
- clear paragraphs
- bullet points
- readable structure

Your task:
- summarize accurately
- explain experiments clearly
- include datasets
- include evaluation methodology
- include findings where available

Do NOT hallucinate.
Do NOT invent information.

If information is incomplete,
explicitly mention it.

================ CONTEXT ================

{context}

================ QUESTION ================

{query}

================ ANSWER ================
"""

In [21]:

# print retrieved chunks

print(context)

The comparative analysis shows that most existing systems focus mainly on accident detection or accident prediction individually. However, they lack an integrated system that combines interactive mapping, accident hotspot visualization, machine learning crash risk prediction, and route safety recommendation.  
The proposed system addresses this gap by integrating machine learning analysis with an interactive map-based interface, enabling users to visualize accident-prone areas, predict crash risk, and choose safer travel routes.  
CERTIFICATE  
This is to certify that the project entitled 'Al Model for Financial Market Prediction'  
is a Bonafide work carried out as part of the course AIM2270, under my guidance from Jan 2025 to May 2025 by Anushka ( 23FE10CAI00419) , student of B. Tech  
(hons.) Computer Science and Engineering (AIML), 4th Semester at the Department of Artificial Intelligence and Machine Learning, Manipal University Jaipur, during the academic semester 4* in partial fu

In [22]:

# LLM RESPONSE

import ollama

print("\nGenerating final answer..\n")

response = ollama.chat(

    model='mistral',

    messages=[
        {
            'role': 'user',
            'content': prompt
        }
    ]
)

final_answer = response['message']['content']

print("\n================ FINAL ANSWER ================\n")

print(final_answer)



Generating final answer..


================ FINAL ANSWER ================

 The title of the project mentioned in the certificate is 'Al Model for Financial Market Prediction'. This project was completed by Anushka, with registration number 23FE10CAI00419. She is a student of B. Tech (hons.) Computer Science and Engineering (AIML), in her 4th semester at the Department of Artificial Intelligence and Machine Learning, Manipal University Jaipur. The project was carried out as part of the course AIM2270 from January 2025 to May 2025 under the guidance of a department supervisor.


In [23]:

# FINAL ANSWER

from IPython.display import display, Markdown

print("\n================ FINAL ANSWER ================\n")

display(Markdown(final_answer))


================ FINAL ANSWER ================



 The title of the project mentioned in the certificate is 'Al Model for Financial Market Prediction'. This project was completed by Anushka, with registration number 23FE10CAI00419. She is a student of B. Tech (hons.) Computer Science and Engineering (AIML), in her 4th semester at the Department of Artificial Intelligence and Machine Learning, Manipal University Jaipur. The project was carried out as part of the course AIM2270 from January 2025 to May 2025 under the guidance of a department supervisor.